In [19]:
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage, HumanMessage, ChatMessage
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import PromptTemplate
from typing import TypedDict


In [20]:
class State(TypedDict):
    joke_title: str
    joke: str
    joke_summary: str

In [21]:
llm = ChatOllama(model="llama3.2:1b", temperature=0)

In [28]:
def createJoke(state: State):
    joke_title = state["joke_title"]

    prompt = PromptTemplate(
        template= """
            You are helpful joke creating AI agent
            You should create a funny joke for this title: {joke_title}
        """,
        input_variables=["joke_title"]
    )

    formatPropmt = prompt.format(joke_title= joke_title)

    res = llm.invoke(formatPropmt)

    return {"joke" : res.content}

def summarizeJoke(state: State):
    joke = state["joke"]

    prompt = PromptTemplate(
        template= """
            You are helpful joke summrize AI agent
            You should create a 300 words summary for this joke: 
            {joke}
            Rule:
                summary should not cross 300  words
        """,
        input_variables=["joke_title"]
    )

    formatPropmt = prompt.format(joke=joke)
    
    res = llm.invoke(formatPropmt)

    return {"joke_summary" : res.content}
    

In [29]:
graph = StateGraph(State)

graph.add_node("createJoke", createJoke)
graph.add_node("summarizeJoke", summarizeJoke)

graph.add_edge(START, "createJoke")
graph.add_edge("createJoke", "summarizeJoke")
graph.add_edge("summarizeJoke", END)

workflow = graph.compile()
workflow.invoke({"joke_title": "phone"})



{'joke_title': 'phone',
 'joke': "Here's one:\n\nWhy did the phone go to therapy?\n\nBecause it had a lot of hang-ups!\n\nHope that one made you laugh!",
 'joke_summary': 'Here\'s a 300-word summary of the joke:\n\nThe joke begins by setting up a familiar scenario, with a phone seeking therapy. However, the punchline subverts expectations by introducing a play on words. Instead of "hang-ups," the phone says it has a lot of "hang-ups." This phrase is a common idiom that means having many problems or issues, but in this case, it\'s also a literal reference to the phone\'s physical structure, with its "hangs" or "hang-ups" being the phone\'s physical components.\n\nThe joke relies on a clever double meaning, where the phrase "hang-ups" is used to describe both the phone\'s emotional state and its physical structure. This creates a humorous contrast between the phone\'s inner turmoil and its outer appearance. The use of this wordplay is a classic example of a "pun," a type of joke that rel